# 🍃 Módulo 3: Clasificación de Patologías en Hojas de Arroz por Visión Artificial
## Proyecto: AgroSmart Perú — Universidad / Software Inteligente

**Objetivo:** Clasificar fotografías de campo de arroz en 3 clases agravadas por el calor y la humedad de El Niño Costero: *Hoja sana*, *Tizón bacteriano (Xanthomonas oryzae)* y *Mancha marrón (Bipolaris oryzae)*.

**Datos reales:** fotografías de campo de 300×300 px del repositorio público de Hugging Face [`girish787/riceLeafDataset`](https://huggingface.co/datasets/girish787/riceLeafDataset) (recopilación de datasets de Kaggle tipo *Paddy Doctor*). Se usan las clases `healthy`, `bacterial_leaf_blight` y `Brown Spot`.

**Pipeline:** imagen → 123 descriptores de color y textura → `HistGradientBoostingClassifier` → probabilidades por clase → recomendación SENASA.


In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

sys.path.append('..')
from src.vision_dataset import preparar_dataset, cargar_metadata, OUT_DIR
from src import vision as V

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
os.makedirs('../Documentos/graficos_evaluacion', exist_ok=True)
print("Librerías importadas correctamente.")


### 1. Descarga y Limpieza del Dataset
`preparar_dataset()` descarga el archivo parquet (~91 MB, solo la primera vez), extrae las 3 clases y aplica dos limpiezas:
1. **Duplicados exactos** (mismo hash perceptual): se conserva una copia; si las copias tienen etiquetas distintas se descartan por ambiguas.
2. **Casi-duplicados** (distancia de Hamming ≤ 6 en un *dHash* de 64 bits): se agrupan. Estos grupos se usan al dividir train/test para que dos fotos casi idénticas nunca queden una en entrenamiento y otra en prueba (eso inflaría artificialmente la exactitud).


In [ ]:
meta = cargar_metadata()   # si no existe data/raw/hojas_arroz/metadata.csv, llama a preparar_dataset()
print(f"Imágenes: {len(meta)} | Grupos de casi-duplicados: {meta['grupo'].nunique()}")
meta['clase'].value_counts().plot(kind='bar', color=['#f0ad4e', '#5cb85c', '#d9534f'], figsize=(6, 3.5), rot=0,
                                  title='Distribución de clases')
plt.tight_layout(); plt.show()


### 2. Inspección Visual de Fotografías de Campo


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(14, 8.5))
for r, clase in enumerate(V.CLASES_ID):
    sub = meta[meta['clase'] == clase].sample(5, random_state=1)
    for c, archivo in enumerate(sub['archivo']):
        axes[r, c].imshow(Image.open(os.path.join(OUT_DIR, archivo)))
        axes[r, c].axis('off')
    axes[r, 0].set_title(V.CLASES_NOMBRE[clase], loc='left', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../Documentos/graficos_evaluacion/04_muestras_patologias_hojas.png', dpi=200)
plt.show()


### 3. Extracción de Características (123 descriptores por imagen)
| Bloque | Nº | Qué captura |
| :--- | :---: | :--- |
| Momentos de color RGB y HSV (media, desviación, asimetría) | 18 | Tono general y variabilidad del follaje |
| Histograma conjunto Tono × Saturación (18×4) | 72 | Proporción de verdes, amarillos y marrones |
| Histograma de brillo (V) | 8 | Zonas secas/blanquecinas vs. sombras |
| Fracción de píxeles verde / amarillo / marrón / blanquecino | 4 | Clorofila, clorosis, necrosis, tejido seco del tizón |
| Textura LBP uniforme (8 vecinos) | 10 | Patrón de manchas y lesiones |
| Histograma de orientación de gradientes + magnitud | 11 | Lesiones alargadas (tizón) vs. puntuales (mancha) |


In [ ]:
X, y = V.construir_matriz(meta, OUT_DIR)
idx_tr, idx_te = V.particion_train_test(meta)
grupos = meta['grupo'].to_numpy()
print(f"Matriz de características: {X.shape}")
print(f"Train: {len(idx_tr)} imágenes | Test: {len(idx_te)} imágenes")
print("Grupos compartidos entre train y test:", len(set(grupos[idx_tr]) & set(grupos[idx_te])))


### 4. Comparación de Modelos (Validación Cruzada por Grupos, 5 particiones, solo sobre Train)


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

candidatos = {
    'Regresión Logística (baseline)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    'SVM RBF': make_pipeline(StandardScaler(), SVC(C=10, gamma='scale')),
    'Random Forest': RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    'HistGradientBoosting': V.nuevo_modelo(),
}
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
filas = []
for nombre, modelo in candidatos.items():
    r = cross_validate(modelo, X[idx_tr], y[idx_tr], groups=grupos[idx_tr], cv=cv,
                       scoring=['accuracy', 'f1_macro'])
    filas.append({'Modelo': nombre,
                  'Accuracy CV': f"{r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f}",
                  'F1 macro CV': f"{r['test_f1_macro'].mean():.3f} ± {r['test_f1_macro'].std():.3f}"})
pd.DataFrame(filas)


### 5. Entrenamiento Final, Evaluación en Test y Exportación
Se entrena el modelo elegido (`HistGradientBoostingClassifier`) con la misma clase de `src/vision.py` que usa la aplicación. Exporta `models/vision_hojas_arroz.pkl` y copia a `data/sample_leaves/` fotos del conjunto de **prueba** para la demo.


In [ ]:
clf = V.PlantDiseaseClassifier()
metricas = clf.train_model()

nombres = [V.CLASES_CORTO[c] for c in V.CLASES_ID]
reporte = pd.DataFrame(metricas['reporte']).T.loc[nombres, ['precision', 'recall', 'f1-score', 'support']]
reporte.round(3)


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
cm = np.array(metricas['matriz_confusion'])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=nombres).plot(ax=ax, cmap='Greens', colorbar=False)
ax.grid(False)
ax.set_title(f"Matriz de confusión en Test (accuracy = {metricas['accuracy']*100:.1f}%)")
plt.tight_layout()
plt.savefig('../Documentos/graficos_evaluacion/06_matriz_confusion_vision.png', dpi=200)
plt.show()


### 6. Inferencia como en la Aplicación (modelo recargado desde disco)


In [ ]:
app_clf = V.PlantDiseaseClassifier()   # carga models/vision_hojas_arroz.pkl al predecir
muestras = sorted(os.listdir('../data/sample_leaves'))
fig, axes = plt.subplots(1, len(muestras), figsize=(3 * len(muestras), 3.4))
for ax, f in zip(axes, muestras):
    ruta = os.path.join('../data/sample_leaves', f)
    res = app_clf.predict_image(ruta)
    ax.imshow(Image.open(ruta)); ax.axis('off')
    ax.set_title(f"{f}\n→ {res['clase_detectada'].split(' (')[0]} ({res['confianza_pct']}%)", fontsize=9)
plt.tight_layout(); plt.show()


### 7. Limitaciones (para la sustentación)
* **Dominio:** el modelo solo vio fotografías de hojas de **arroz** (en campo o sobre fondo blanco). No es válido para limón ni para dibujos: en esas imágenes puede equivocarse con probabilidades altas.
* **Sesgo de fondo:** el 29% de las hojas sanas y el 15% de las de mancha marrón están fotografiadas sobre fondo blanco, pero **ninguna** de tizón bacteriano. En test, el modelo acierta 98.1% en fotos con fondo blanco y 94.8% en fotos de campo, así que no se limita a usar el fondo; aun así, una hoja con tizón fotografiada sobre fondo blanco probablemente se clasificaría mal.
* **Probabilidades poco calibradas:** el *boosting* tiende a dar probabilidades cercanas a 100%; la confianza mostrada no debe leerse como certeza.
* **Origen de las fotos:** las imágenes no son de Piura; pertenecen a datasets públicos internacionales. Un despliegue real requeriría fotos locales etiquetadas por SENASA.
* **Posible sesgo de sesión:** si las fotos de cada clase se tomaron en campañas distintas, el modelo puede aprender parte de la iluminación o del encuadre además del síntoma.

### 8. Directrices Técnicas Asociadas (SENASA)
* **Hoja Sana:** Mantener monitoreo preventivo y descolmatación de drenes.
* **Tizón Bacteriano (*Xanthomonas*):** Aplicar bactericidas a base de cobre y suspender fertilización con nitrógeno.
* **Mancha Marrón (*Bipolaris*):** Drenar parcelas encharcadas y aplicar fungicidas sistémicos (Triazoles).
